# YOLOv8n fine-tune on VisDrone persons (Colab)

Put `prepare_visdrone.py`, `train.py` and this notebook in `MyDrive/security_drone`.
For hard negatives, also put the Gazebo frame folders in `MyDrive/security_drone/negatives/`
(e.g. `negatives/hard_negatives/*.jpg`).
Runtime → Change runtime type → **L4 GPU** (T4 also works), then run the cells in order.
Weights are written to `MyDrive/security_drone/runs/<RUN>`, so a disconnect loses at most one epoch.

In [ ]:
DRIVE = "/content/drive/MyDrive/security_drone"
# First run: RUN = "visdrone_person", NEGATIVES = "".
RUN = "visdrone_person_neg"
NEGATIVES = "hard_negatives"   # folders under negatives/ (training/select_negatives.py)

NEG_FLAG = " ".join(f"{DRIVE}/negatives/{n}" for n in NEGATIVES.split())
NEG_FLAG = f"--negatives {NEG_FLAG}" if NEG_FLAG else ""
DATA = "/content/datasets/" + ("visdrone_person_neg" if NEGATIVES else "visdrone_person") + "/data.yaml"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q ultralytics==8.4.183
%cd /content

Download VisDrone (~2 GB) to the Colab disk and build the single-class set. Check the counts: train 6471 VisDrone images, 106396 person boxes, plus the hard negatives if any.

In [ ]:
!python {DRIVE}/prepare_visdrone.py --root /content/datasets {NEG_FLAG}

Train. 1280 px matches the companion's inference size. In the first lines check `AMP: checks passed` and `0 corrupt`.

In [ ]:
!python {DRIVE}/train.py --data {DATA} --imgsz 1280 --batch 8 --workers 2 --project {DRIVE}/runs --name {RUN}

**Only after a disconnect:** run the settings, mount, pip and dataset cells again, then this one.

In [ ]:
!python {DRIVE}/train.py --data {DATA} --imgsz 1280 --batch 8 --workers 2 --project {DRIVE}/runs --name {RUN} --resume